# Setup

## Imports

In [1]:
!pip freeze

asttokens==3.0.1
comm==0.2.3
contourpy==1.3.2
cycler==0.12.1
debugpy==1.8.21
decorator==5.3.1
exceptiongroup==1.3.1
executing==2.2.1
fonttools==4.63.0
ipykernel==7.2.0
ipython==8.39.0
jedi==0.20.0
jupyter_client==8.9.0
jupyter_core==5.9.1
kiwisolver==1.5.0
matplotlib==3.10.9
matplotlib-inline==0.2.2
nest-asyncio==1.6.0
numpy==2.2.6
packaging==26.2
pandas==2.3.3
parso==0.8.7
pexpect==4.9.0
pillow==12.2.0
platformdirs==4.10.0
prompt_toolkit==3.0.52
psutil==7.2.2
ptyprocess==0.7.0
pure_eval==0.2.3
pyarrow==24.0.0
Pygments==2.20.0
pyparsing==3.3.2
python-dateutil==2.9.0.post0
pytz==2026.2
pyzmq==27.1.0
six==1.17.0
stack-data==0.6.3
tornado==6.5.6
traitlets==5.15.1
typing_extensions==4.15.0
tzdata==2026.2
wcwidth==0.8.0


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gc
from collections import defaultdict

## Variables Globales

In [2]:
dataset_path = "global_concatenado/global_concatenado.CSV"

## Carga de Dataset

In [4]:
def infer_optimal_dtypes(
    csv_path,
    chunksize=100_000,
    max_rows=None,
    low_memory=False
):
    """
    First pass over CSV:
    - Reads in chunks
    - Detects optimal dtype for each column
    - Handles ints, floats, bools, strings, categories, datetimes

    Returns:
        dtype_map: dict for pandas dtype loading
        parse_dates: list of datetime columns
    """

    stats = defaultdict(lambda: {
        "dtype_candidates": set(),
        "min": None,
        "max": None,
        "n_unique_sample": set(),
        "non_null": 0,
    })

    rows_read = 0

    reader = pd.read_csv(
        csv_path,
        chunksize=chunksize,
        low_memory=low_memory
    )

    for chunk in reader:

        if max_rows is not None:
            remaining = max_rows - rows_read
            if remaining <= 0:
                break
            chunk = chunk.iloc[:remaining]

        rows_read += len(chunk)

        for col in chunk.columns:
            s = chunk[col].dropna()

            if len(s) == 0:
                continue

            info = stats[col]

            # Try bool detection
            unique_vals = set(s.unique())

            bool_like = unique_vals.issubset(
                {0, 1, True, False, "0", "1", "True", "False", "true", "false"}
            )

            if bool_like:
                info["dtype_candidates"].add("bool")
                continue

            # Integer columns
            if pd.api.types.is_integer_dtype(s):

                info["dtype_candidates"].add("int")

                col_min = s.min()
                col_max = s.max()

                info["min"] = (
                    col_min if info["min"] is None
                    else min(info["min"], col_min)
                )

                info["max"] = (
                    col_max if info["max"] is None
                    else max(info["max"], col_max)
                )

                continue

            # Float columns
            if pd.api.types.is_float_dtype(s):

                info["dtype_candidates"].add("float")

                col_min = s.min()
                col_max = s.max()

                info["min"] = (
                    col_min if info["min"] is None
                    else min(info["min"], col_min)
                )

                info["max"] = (
                    col_max if info["max"] is None
                    else max(info["max"], col_max)
                )

                continue

            # Datetime detection (sample-based)
            if len(s) > 0:
                sample = s.iloc[:50]

                try:
                    parsed = pd.to_datetime(sample, errors="raise")

                    if parsed.notna().mean() > 0.95:
                        info["dtype_candidates"].add("datetime")
                        continue

                except Exception:
                    pass

            # String / category detection
            info["dtype_candidates"].add("object")

            # Sample uniques for category decision
            sample_uniques = set(s.astype(str).unique()[:5000])
            info["n_unique_sample"].update(sample_uniques)
            info["non_null"] += len(s)

    # Decide final dtypes
    dtype_map = {}
    parse_dates = []

    for col, info in stats.items():

        candidates = info["dtype_candidates"]

        # Boolean
        if candidates == {"bool"}:
            dtype_map[col] = "boolean"
            continue

        # Integer
        if "int" in candidates and "float" not in candidates:

            min_val = info["min"]
            max_val = info["max"]

            if min_val >= 0:
                if max_val <= np.iinfo(np.uint8).max:
                    dtype_map[col] = np.uint8
                elif max_val <= np.iinfo(np.uint16).max:
                    dtype_map[col] = np.uint16
                elif max_val <= np.iinfo(np.uint32).max:
                    dtype_map[col] = np.uint32
                else:
                    dtype_map[col] = np.uint64

            else:
                if (
                    min_val >= np.iinfo(np.int8).min
                    and max_val <= np.iinfo(np.int8).max
                ):
                    dtype_map[col] = np.int8

                elif (
                    min_val >= np.iinfo(np.int16).min
                    and max_val <= np.iinfo(np.int16).max
                ):
                    dtype_map[col] = np.int16

                elif (
                    min_val >= np.iinfo(np.int32).min
                    and max_val <= np.iinfo(np.int32).max
                ):
                    dtype_map[col] = np.int32

                else:
                    dtype_map[col] = np.int64

            continue

        # Float
        if "float" in candidates:
            min_val = info["min"]
            max_val = info["max"]

            # Float32 unless too large
            float32_limit = np.finfo(np.float32).max

            if (
                abs(min_val) < float32_limit
                and abs(max_val) < float32_limit
            ):
                dtype_map[col] = np.float32
            else:
                dtype_map[col] = np.float64

            continue

        # Datetime
        if "datetime" in candidates:
            parse_dates.append(col)
            continue

        # String/category
        if "object" in candidates:

            unique_count = len(info["n_unique_sample"])
            total_count = max(info["non_null"], 1)

            # Heuristic:
            # low cardinality => category
            if unique_count / total_count < 0.2:
                dtype_map[col] = "category"
            else:
                dtype_map[col] = "string"

    return dtype_map, parse_dates


def load_csv_optimized(
    csv_path,
    chunksize=100_000,
    max_rows=None
):
    """
    Optimized CSV loading using inferred dtypes.

    Returns:
        pandas DataFrame
    """

    print("Inferring dtypes...")
    dtype_map, parse_dates = infer_optimal_dtypes(
        csv_path,
        chunksize=chunksize,
        max_rows=max_rows
    )

    print("Chosen dtypes:")
    for k, v in dtype_map.items():
        print(f"{k}: {v}")

    chunks = []
    rows_read = 0

    reader = pd.read_csv(
        csv_path,
        chunksize=chunksize,
        dtype=dtype_map,
        parse_dates=parse_dates,
        low_memory=False
    )

    for chunk in reader:

        if max_rows is not None:
            remaining = max_rows - rows_read
            if remaining <= 0:
                break
            chunk = chunk.iloc[:remaining]

        rows_read += len(chunk)
        chunks.append(chunk)

    df = pd.concat(chunks, ignore_index=True)

    return df

In [16]:
df = load_csv_optimized(dataset_path)

Inferring dtypes...


/tmp/ipykernel_29206/1611202921.py:108: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(sample, errors="raise")
/tmp/ipykernel_29206/1611202921.py:108: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(sample, errors="raise")
/tmp/ipykernel_29206/1611202921.py:108: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(sample, errors="raise")
/tmp/ipykernel_29206/1611202921.py:108: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consist

Chosen dtypes:
Unnamed: 0.1: <class 'numpy.uint32'>
Unnamed: 0: <class 'numpy.uint16'>
source: category
psi_psa1: <class 'numpy.float32'>
psi_psa2: <class 'numpy.float32'>
psi_psa3: <class 'numpy.float32'>
psi_psa4: <class 'numpy.float32'>
psi_tablero: <class 'numpy.float32'>
flujo: <class 'numpy.float32'>
totalizador: <class 'numpy.float32'>
r_psa1: boolean
r_psa2: boolean
r_psa3: boolean
r_psa4: boolean
r_gen1: boolean
r_gen2: boolean
r_bar: boolean
r_sec1: boolean
r_sec2: boolean
r_com1: boolean
r_com2: boolean
r_com3: boolean
r_com4: boolean
sp_s1: <class 'numpy.float32'>
sp_s2: <class 'numpy.float32'>
sp_s3: <class 'numpy.float32'>
sp_s4: <class 'numpy.float32'>
sp_s5: <class 'numpy.float32'>
sp_s6: <class 'numpy.float32'>
sp_s7: <class 'numpy.float32'>
sp_s8: <class 'numpy.float32'>
sp_s9: <class 'numpy.float32'>
sp_s10: <class 'numpy.float32'>
sp_s11: <class 'numpy.float32'>
sp_s12: <class 'numpy.float32'>
hb_gen1: <class 'numpy.float32'>
hb_gen2: boolean
hb_sec1: <class 'numpy.

In [31]:
def save_dataframe_parquet(
    df: pd.DataFrame,
    output_path="dataset.parquet",
    compression="snappy",
    index=False
):
    """
    Save a DataFrame to Parquet.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame to save.

    output_path : str
        Output parquet file path.

    compression : str
        Compression type:
        - 'snappy' (fast, default)
        - 'gzip'   (smaller file, slower)
        - 'brotli' (very compact)
        - 'zstd'   (excellent balance)
        - None     (no compression)

    index : bool
        Whether to save DataFrame index.
    """

    df.to_parquet(
        output_path,
        engine="pyarrow",   # recommended
        compression=compression,
        index=index
    )

    print(f"Saved parquet to: {output_path}")

def load_parquet_df(parquet_path, columns=None):
    """
    Load a DataFrame from parquet.

    Parameters
    ----------
    parquet_path : str
        Path to parquet file.

    columns : list[str] | None
        Optional subset of columns to load.

    Returns
    -------
    pd.DataFrame
    """

    df = pd.read_parquet(
        parquet_path,
        engine="pyarrow",
        columns=columns
    )

    print(f"Loaded dataframe: {df.shape}")
    return df

In [ ]:
save_dataframe_parquet(
    df,
    "huge_dataset_non_clean.parquet"
)

Saved parquet to: huge_dataset_non_clean.parquet


In [55]:
df = load_parquet_df("huge_dataset_non_clean.parquet")

Loaded dataframe: (8644192, 109)


In [56]:
summarize_df(df)

['Unnamed: 0.1', 'Unnamed: 0', 'source', 'psi_psa1', 'psi_psa2', 'psi_psa3', 'psi_psa4', 'psi_tablero', 'flujo', 'totalizador', 'TIME', 'r_psa1', 'r_psa2', 'r_psa3', 'r_psa4', 'r_gen1', 'r_gen2', 'r_bar', 'r_sec1', 'r_sec2', 'r_com1', 'r_com2', 'r_com3', 'r_com4', 'sp_s1', 'sp_s2', 'sp_s3', 'sp_s4', 'sp_s5', 'sp_s6', 'sp_s7', 'sp_s8', 'sp_s9', 'sp_s10', 'sp_s11', 'sp_s12', 'hb_gen1', 'hb_gen2', 'hb_sec1', 'hb_sec2', 'hb_com1', 'hb_com2', 'hb_com3', 'hb_com4', 'hb_psa1', 'hb_psa2', 'hb_psa3', 'hb_psa4', 'rst_gen1', 'rst_gen2', 'rst_sec1', 'rst_sec2', 'rst_com1', 'rst_com2', 'rst_com3', 'rst_com4', 'rst_psa1', 'rst_psa2', 'rst_psa3', 'rst_psa4', 'hb_s1', 'hb_s2', 'hb_s3', 'hb_s4', 'hb_s5', 'hb_s6', 'hb_s7', 'hb_s8', 'hb_s9', 'hb_s10', 'hb_s11', 'hb_s12', 'ox_s1', 'ox_s2', 'ox_s3', 'ox_s4', 'ox_s5', 'ox_s6', 'ox_s7', 'ox_s8', 'ox_s9', 'ox_s10', 'ox_s11', 'ox_s12', 'm_s1', 'm_s2', 'm_s3', 'm_s4', 'm_s5', 'm_s6', 'm_s7', 'm_s8', 'm_s9', 'm_s10', 'm_s11', 'm_s12', 'mb_g1_temperatura_f', 'mb_

## Limpieza de Dataset

### PASO 1 - Eliminar columnas basura

In [57]:
columnas_basura = ['Unnamed: 0.1', 'Unnamed: 0', 'Sistema']
df.drop(columns=[c for c in columnas_basura if c in df.columns], inplace=True)

### PASO 2 - Eliminar pontones con arquitectura inválida

Los pontones POX1, POX47, POX64, POX66 y POX67 reportaron arquitecturas
imposibles (3 compresores + 4 PSA) o señales fantasma persistentes en hb_com4.

In [58]:
pontones_invalidos = ['POX1', 'POX47', 'POX64', 'POX66', 'POX67']
indices_drop = df[df['source'].isin(pontones_invalidos)].index
df.drop(index=indices_drop, inplace=True)

### PASO 4 — Separar en dos dataframes según arquitectura
- **4 compresores → 4 PSA**: detectados porque tienen datos en `r_com4`
- **3 compresores → 6 PSA**: el resto

In [59]:
sources_4comp = df.dropna(subset=['r_com4'])['source'].unique()

df_4comp = df[df['source'].isin(sources_4comp)].copy()
df_3comp = df[~df['source'].isin(sources_4comp)].copy()

del df
gc.collect()

print(f'4 Compresores: {len(df_4comp):,} filas')
print(f'3 Compresores: {len(df_3comp):,} filas')

4 Compresores: 5,862,409 filas
3 Compresores: 1,995,871 filas


### PASO 5 - Purga de columnas que no corresponden a cada arquitectura
- df_4comp: no tiene PSA 5 ni PSA 6 → se eliminan
- df_3comp: no tiene compresor 4 → se eliminan

In [60]:
cols_psa56  = [c for c in df_4comp.columns if 'psa5' in c or 'psa6' in c]
cols_com4   = [c for c in df_3comp.columns if 'com4' in c]

df_4comp.drop(columns=cols_psa56, inplace=True, errors='ignore')
df_3comp.drop(columns=cols_com4,  inplace=True, errors='ignore')

print(f'df_4comp: {df_4comp.shape[1]} columnas (eliminadas {len(cols_psa56)} de PSA5/6)')
print(f'df_3comp: {df_3comp.shape[1]} columnas (eliminadas {len(cols_com4)} de COM4)')

df_4comp: 98 columnas (eliminadas 8 de PSA5/6)
df_3comp: 103 columnas (eliminadas 3 de COM4)


### Paso 6 - Manejo de nulos

In [61]:
cols_ox = [f'ox_s{i}' for i in range(1, 13)]


def reportar_apagones_prolongados(
    df,
    nombre,
    min_minutes=5
):
    print(f'\n⏳ Analizando {nombre}...')

    # Work on a copy to avoid modifying original df
    temp_df = df.copy()

    temp_df['TIME'] = pd.to_datetime(
        temp_df['TIME'],
        errors='coerce'
    )

    temp_df.sort_values(
        by=['source', 'TIME'],
        inplace=True
    )

    cols_ox_presentes = [
        c for c in cols_ox
        if c in temp_df.columns
    ]

    # Detect shutdown rows
    temp_df['es_apagon'] = (
        temp_df[cols_ox_presentes]
        .isnull()
        .all(axis=1)
    )

    resultados = {}
    total_apagones = 0

    # Analyze per source
    for source, group in temp_df.groupby('source'):

        group = group.copy()

        # Identify contiguous shutdown blocks
        group['bloque'] = (
            group['es_apagon']
            != group['es_apagon'].shift()
        ).cumsum()

        apagones_prolongados = 0

        # Only shutdown periods
        shutdown_blocks = group[group['es_apagon']]

        for _, shutdown in shutdown_blocks.groupby('bloque'):

            if len(shutdown) < 2:
                continue

            start = shutdown['TIME'].iloc[0]
            end = shutdown['TIME'].iloc[-1]

            duration_minutes = (
                (end - start)
                .total_seconds()
                / 60
            )

            if duration_minutes > min_minutes:
                apagones_prolongados += 1

        resultados[source] = apagones_prolongados
        total_apagones += apagones_prolongados

    print(f'✅ Total prolonged shutdowns (> {min_minutes} min): {total_apagones:,}')
    print('\nPor source:')

    for source, count in sorted(resultados.items()):
        print(f'  {source}: {count:,}')

    return resultados

In [62]:
apagones_4comp = reportar_apagones_prolongados(
    df_4comp,
    'Sistema 4 Compresores'
)

apagones_3comp = reportar_apagones_prolongados(
    df_3comp,
    'Sistema 3 Compresores'
)


⏳ Analizando Sistema 4 Compresores...
✅ Total prolonged shutdowns (> 5 min): 47

Por source:
  POX10: 0
  POX11: 1
  POX13: 0
  POX15: 0
  POX16: 0
  POX17: 0
  POX18: 0
  POX19: 1
  POX20: 0
  POX21: 1
  POX22: 1
  POX23: 20
  POX24: 0
  POX25: 1
  POX26: 0
  POX27: 0
  POX28: 0
  POX29: 1
  POX3: 0
  POX30: 3
  POX31: 0
  POX32: 0
  POX33: 0
  POX34: 1
  POX35: 1
  POX39: 1
  POX41: 2
  POX43: 0
  POX44: 0
  POX45: 0
  POX49: 1
  POX53: 0
  POX54: 0
  POX56: 2
  POX57: 0
  POX58: 1
  POX6: 0
  POX62: 1
  POX8: 2
  POX9: 5
  POXC1: 1

⏳ Analizando Sistema 3 Compresores...
✅ Total prolonged shutdowns (> 5 min): 8

Por source:
  POX37: 0
  POX38: 0
  POX40: 2
  POX42: 1
  POX46: 0
  POX48: 1
  POX50: 2
  POX51: 0
  POX55: 0
  POX59: 1
  POX60: 0
  POX61: 0
  POX63: 0
  POX65: 1


In [19]:
dfs_by_source = {
    source: group.copy()
    for source, group in df.groupby("source")
}

In [20]:
for key in dfs_by_source.keys():
    pass
    print("="*80)
    print(key)
    summarize_df(dfs_by_source[key])

POX1
['Unnamed: 0.1', 'Unnamed: 0', 'source', 'psi_psa1', 'psi_psa2', 'psi_psa3', 'psi_psa4', 'psi_tablero', 'flujo', 'totalizador', 'TIME', 'r_psa1', 'r_psa2', 'r_psa3', 'r_psa4', 'r_gen1', 'r_gen2', 'r_bar', 'r_sec1', 'r_sec2', 'r_com1', 'r_com2', 'r_com3', 'r_com4', 'sp_s1', 'sp_s2', 'sp_s3', 'sp_s4', 'sp_s5', 'sp_s6', 'sp_s7', 'sp_s8', 'sp_s9', 'sp_s10', 'sp_s11', 'sp_s12', 'hb_gen1', 'hb_gen2', 'hb_sec1', 'hb_sec2', 'hb_com1', 'hb_com2', 'hb_com3', 'hb_com4', 'hb_psa1', 'hb_psa2', 'hb_psa3', 'hb_psa4', 'rst_gen1', 'rst_gen2', 'rst_sec1', 'rst_sec2', 'rst_com1', 'rst_com2', 'rst_com3', 'rst_com4', 'rst_psa1', 'rst_psa2', 'rst_psa3', 'rst_psa4', 'hb_s1', 'hb_s2', 'hb_s3', 'hb_s4', 'hb_s5', 'hb_s6', 'hb_s7', 'hb_s8', 'hb_s9', 'hb_s10', 'hb_s11', 'hb_s12', 'ox_s1', 'ox_s2', 'ox_s3', 'ox_s4', 'ox_s5', 'ox_s6', 'ox_s7', 'ox_s8', 'ox_s9', 'ox_s10', 'ox_s11', 'ox_s12', 'm_s1', 'm_s2', 'm_s3', 'm_s4', 'm_s5', 'm_s6', 'm_s7', 'm_s8', 'm_s9', 'm_s10', 'm_s11', 'm_s12', 'mb_g1_temperatura_f',

In [21]:
print(len(dfs_by_source.keys()))

60
